# 10 — LSTM Signal Exploration

**Question:** Can you read rebuild stage from the VV curve?

Before training any model, we need to visually verify that the SAR time series
actually encodes rebuild stage information. If a human cannot see the pattern,
an LSTM probably cannot learn it either.

This notebook normalizes VV backscatter relative to the pre-fire baseline,
plots representative examples per rebuild stage, and runs unsupervised
clustering to see if natural groups align with permit-derived labels.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Observation dates as datetime objects for plotting
DATE_LABELS = ["2021-11", "2022-01", "2022-06", "2023-06", "2024-06"]
DATE_OBJECTS = [datetime.strptime(d, "%Y-%m") for d in DATE_LABELS]

# Rebuild stage names and colors
STAGE_NAMES = ["destroyed", "cleared_lot", "framing", "structure_substantial", "rebuild_complete"]
STAGE_COLORS = {
    "destroyed": "#D32F2F",
    "cleared_lot": "#FF9800",
    "framing": "#FFC107",
    "structure_substantial": "#4CAF50",
    "rebuild_complete": "#1565C0",
}
print("Libraries loaded.")

## Load VV time series for destroyed parcels

The zonal stats file has per-parcel mean VV (dB) for each observation date.
We normalize by subtracting each parcel's Nov 2021 (pre-fire) baseline value,
so 0 = same as pre-fire, negative = less backscatter than before.

In [ ]:
# ----- Load zonal stats -----
zonal = pd.read_parquet(DATA_DIR / "tabular" / "sar_zonal_stats.parquet")
print(f"Zonal stats shape: {zonal.shape}")
print(f"Columns: {list(zonal.columns)}")
print(f"Unique dates: {sorted(zonal['date'].unique())}")
print(f"Unique parcels: {zonal['parcel_id'].nunique()}")

# ----- Filter to VV mean values and pivot to wide format -----
# Each row = one parcel, columns = VV mean per date
vv_stats = zonal[zonal["raster"].str.contains("vv", case=False)].copy()
vv_pivot = vv_stats.pivot_table(
    index="parcel_id", columns="date", values="mean", aggfunc="first"
).sort_index(axis=1)

print(f"\nVV pivot shape: {vv_pivot.shape}")
print(vv_pivot.head())

# ----- Normalize: subtract pre-fire baseline (Nov 2021) -----
baseline_col = [c for c in vv_pivot.columns if "2021" in str(c)][0]
baseline = vv_pivot[baseline_col]

vv_norm = vv_pivot.subtract(baseline, axis=0)
print("\nNormalized VV (0 = pre-fire baseline):")
print(vv_norm.describe().round(2))

# ----- Stack into matrix for ML -----
# Drop parcels with any NaN
vv_norm_clean = vv_norm.dropna()
vv_matrix = vv_norm_clean.values  # (n_parcels, 5 timesteps)
parcel_ids = vv_norm_clean.index.values
print(f"\nClean matrix: {vv_matrix.shape} (parcels x timesteps)")

# ----- Load labels -----
labels_df = pd.read_parquet(DATA_DIR / "tabular" / "parcel_labels.parquet")
# Merge stage labels onto our parcel set
label_map = labels_df.set_index("parcel_id")["rebuild_stage"] if "rebuild_stage" in labels_df.columns else None
if label_map is not None:
    stages = pd.Series(parcel_ids).map(label_map).values
    print(f"Labeled parcels: {pd.notna(stages).sum()} / {len(stages)}")
else:
    stages = np.full(len(parcel_ids), np.nan)
    print("No rebuild_stage column found — will use damage status only.")

## Plot representative examples by rebuild stage

Expected VV signatures:
- **destroyed**: sharp dip at Jan 2022, stays low
- **cleared_lot**: drops even BELOW post-fire (flat concrete is more specular than rubble)
- **framing**: recovering toward baseline by 2023
- **rebuild_complete**: back to (or above) pre-fire baseline by 2024

Top row: parcels with known stage labels. Bottom row: unlabeled parcels for blind guessing.

In [ ]:
def plot_vv_curve(ax, vv_values, title, stage=None, color="#333333"):
    """Plot a single parcel's normalized VV time series."""
    fill_color = STAGE_COLORS.get(stage, "#90CAF9")

    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5, label="Pre-fire baseline")
    ax.fill_between(DATE_OBJECTS, 0, vv_values, alpha=0.2, color=fill_color)
    ax.plot(DATE_OBJECTS, vv_values, "o-", color=color, linewidth=2, markersize=6)

    ax.set_title(title, fontsize=10)
    ax.set_ylabel("VV dB (norm)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(vv_matrix.min() - 0.5, vv_matrix.max() + 0.5)


fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# ----- Top row: known-stage examples -----
for col_idx, stage_name in enumerate(STAGE_NAMES[:4]):
    ax = axes[0, col_idx]

    # Find a parcel matching this stage
    if label_map is not None:
        stage_mask = (stages == stage_name)
        if stage_mask.any():
            idx = np.where(stage_mask)[0][0]
        else:
            # Fallback: use index position
            idx = col_idx * (len(vv_matrix) // 5)
    else:
        idx = col_idx * (len(vv_matrix) // 5)

    plot_vv_curve(
        ax, vv_matrix[idx],
        f"Known: {stage_name}\n(parcel {parcel_ids[idx]})",
        stage=stage_name,
        color=STAGE_COLORS.get(stage_name, "#333"),
    )

# ----- Bottom row: unknown parcels for blind guessing -----
# Pick 4 random parcels without labels (or random if all labeled)
rng = np.random.RandomState(42)
if label_map is not None:
    unlabeled_mask = pd.isna(stages)
    if unlabeled_mask.sum() >= 4:
        unknown_indices = rng.choice(np.where(unlabeled_mask)[0], size=4, replace=False)
    else:
        unknown_indices = rng.choice(len(vv_matrix), size=4, replace=False)
else:
    unknown_indices = rng.choice(len(vv_matrix), size=4, replace=False)

for col_idx, idx in enumerate(unknown_indices):
    ax = axes[1, col_idx]
    true_stage = stages[idx] if pd.notna(stages[idx]) else "unknown"
    plot_vv_curve(
        ax, vv_matrix[idx],
        f"Guess this one\n(parcel {parcel_ids[idx]})",
        stage=None,
        color="#555555",
    )
    # Hidden annotation — uncomment to reveal true stage
    # ax.annotate(f"Answer: {true_stage}", xy=(0.5, 0.02),
    #             xycoords='axes fraction', ha='center', fontsize=8, color='red')

axes[0, 0].set_ylabel("Known stage\nVV dB (norm)", fontsize=11)
axes[1, 0].set_ylabel("Unknown stage\nVV dB (norm)", fontsize=11)

fig.suptitle(
    "VV Backscatter Time Series by Rebuild Stage",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(DATA_DIR / "results" / "vv_rebuild_signatures.png", dpi=150, bbox_inches="tight")
plt.show()

## Unsupervised clustering -- do natural clusters align with rebuild stages?

If KMeans on the raw VV trajectories produces clusters that map to rebuild
stages, then the signal is strong enough for supervised learning.

In [ ]:
# ----- Scale + PCA -----
scaler = StandardScaler()
vv_scaled = scaler.fit_transform(vv_matrix)

pca = PCA(n_components=2, random_state=42)
vv_pca = pca.fit_transform(vv_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.round(3)}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.1%}")

# ----- KMeans k=4 -----
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(vv_scaled)

# ----- Plot -----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: colored by KMeans cluster
cluster_colors = ["#D32F2F", "#FF9800", "#4CAF50", "#1565C0"]
for k in range(4):
    mask = cluster_labels == k
    ax1.scatter(
        vv_pca[mask, 0], vv_pca[mask, 1],
        c=cluster_colors[k], label=f"Cluster {k} (n={mask.sum()})",
        alpha=0.6, s=30, edgecolors="white", linewidth=0.3,
    )
ax1.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax1.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax1.set_title("KMeans Clusters (k=4)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: colored by permit-derived stage (if available)
if label_map is not None:
    for stage_name in STAGE_NAMES:
        mask = stages == stage_name
        if mask.any():
            ax2.scatter(
                vv_pca[mask, 0], vv_pca[mask, 1],
                c=STAGE_COLORS[stage_name], label=f"{stage_name} (n={mask.sum()})",
                alpha=0.6, s=30, edgecolors="white", linewidth=0.3,
            )
    # Plot unlabeled in gray
    unlabeled = pd.isna(stages)
    if unlabeled.any():
        ax2.scatter(
            vv_pca[unlabeled, 0], vv_pca[unlabeled, 1],
            c="#BDBDBD", label=f"unlabeled (n={unlabeled.sum()})",
            alpha=0.3, s=15,
        )
    ax2.set_title("Permit-Derived Stage Labels")
else:
    ax2.scatter(vv_pca[:, 0], vv_pca[:, 1], c="#BDBDBD", alpha=0.3, s=15)
    ax2.set_title("No stage labels available")

ax2.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax2.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle("Unsupervised Clustering of VV Trajectories", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ----- Print cluster composition -----
if label_map is not None:
    print("\nCluster composition vs permit-derived labels:")
    comp_df = pd.DataFrame({"cluster": cluster_labels, "stage": stages})
    ct = pd.crosstab(comp_df["cluster"], comp_df["stage"], margins=True)
    print(ct.to_string())

## Observations

Expected findings from the clustering analysis:

1. **rebuild_complete** parcels should separate clearly — their VV trajectory
   recovers to (or above) pre-fire baseline, which is a distinctive shape in
   the 5-timestep feature space.

2. **cleared_lot** vs **framing** (foundation stage) parcels look similar in SAR
   because both are low-backscatter surfaces (concrete slab vs partial framing).
   The LSTM in notebook 11 will need the temporal *direction* (still dropping
   vs starting to rise) to distinguish them.

3. **destroyed** parcels (still rubble) have a characteristic shape: sharp dip
   at Jan 2022, then flat or slowly declining — rubble scatters diffusely.

4. The unsupervised clusters will likely show 2-3 clean groups, not 4-5,
   because the intermediate rebuild stages overlap in the SAR signal space.
   This motivates adding parcel context features (area, pixel count) in the
   LSTM model.